# Q2 — 평가 · Grad-CAM · 추론 데모
Q1에서 저장한 `models/checkpoints/baseline_resnet18.pt`를 불러와 성능 재확인, **Grad-CAM으로 근거 시각화**, 단일 웨이퍼 추론까지.
필요 패키지: `opencv` 불필요. `scikit-learn`만 있으면 됨.

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p = Path.cwd()
for cand in [_p, _p.parent, _p.parent.parent]:
    if (cand / "utils").is_dir():
        sys.path.insert(0, str(cand)); PROJ = cand; break
else:
    PROJ = _p
from utils.korean_font import set_korean_font
set_korean_font()

import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM_CLASSES = len(CLASSES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROC = PROJ/"data"/"processed"; CKPT = PROJ/"models"/"checkpoints"/"baseline_resnet18.pt"
FIG_DIR = PROJ/"results"/"figures"/"gradcam"; FIG_DIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| ckpt 존재:", CKPT.exists())

In [ ]:
# --- 체크포인트 + test 데이터 로드 ---
ck = torch.load(CKPT, map_location=DEVICE)
net = models.resnet18(weights=None)
net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES)
net.load_state_dict(ck["model"]); net.to(DEVICE).eval()

d = np.load(PROC/"wafer_test.npz", allow_pickle=True)
Xte, yte = d["X"], d["y"]

def to_tensor(img_u8):
    img = img_u8.astype(np.float32)/2.0
    img = (img-0.5)/0.5
    img = np.expand_dims(img,0).repeat(3,0)
    return torch.from_numpy(img).unsqueeze(0)
print("test:", Xte.shape)

In [ ]:
# --- 전체 지표 재확인 ---
from sklearn.metrics import classification_report, accuracy_score, f1_score
preds=[]
with torch.no_grad():
    for i in range(0, len(Xte), 512):
        batch = np.stack([ (Xte[j].astype(np.float32)/2.0 -0.5)/0.5 for j in range(i, min(i+512,len(Xte))) ])
        batch = np.expand_dims(batch,1).repeat(3,1)
        out = net(torch.from_numpy(batch).to(DEVICE))
        preds.append(out.argmax(1).cpu().numpy())
pt = np.concatenate(preds)
print(f"accuracy={accuracy_score(yte,pt):.3f} | macro-F1={f1_score(yte,pt,average='macro'):.3f}\n")
print(classification_report(yte, pt, target_names=CLASSES, digits=3, zero_division=0))

In [ ]:
# --- Grad-CAM 구현 (layer4 타깃) ---
class GradCAM:
    def __init__(self, model, layer):
        self.model = model; self.acts=None; self.grads=None
        layer.register_forward_hook(self._fwd)
        layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.acts = o.detach()
    def _bwd(self, m, gi, go): self.grads = go[0].detach()
    def __call__(self, x, cls=None):
        self.model.zero_grad()
        out = self.model(x)
        if cls is None: cls = out.argmax(1).item()
        out[0, cls].backward()
        w = self.grads.mean(dim=(2,3), keepdim=True)        # GAP of grads
        cam = F.relu((w*self.acts).sum(1)).squeeze().cpu().numpy()
        cam = (cam - cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, cls, out.softmax(1)[0, cls].item()

cam_engine = GradCAM(net, net.layer4)
print("Grad-CAM 준비 완료 (target=layer4)")

In [ ]:
# --- 맞춘/틀린 샘플 Grad-CAM 시각화 ---
rng = np.random.default_rng(0)
# defect(=none 아닌) 샘플 위주로 8개 뽑기
defect_idx = np.where(yte != 8)[0]
pick = rng.choice(defect_idx, size=8, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(20,5))
for col, idx in enumerate(pick):
    x = to_tensor(Xte[idx]).to(DEVICE)
    cam, pred, prob = cam_engine(x, cls=None)
    base = Xte[idx]/2.0
    axes[0,col].imshow(base, cmap="gray"); axes[0,col].axis("off")
    ok = "O" if pred==yte[idx] else "X"
    axes[0,col].set_title(f"실제:{CLASSES[yte[idx]]}\n예측:{CLASSES[pred]} {ok}", fontsize=8)
    axes[1,col].imshow(base, cmap="gray")
    axes[1,col].imshow(cam, cmap="jet", alpha=0.5,
                       extent=(0,128,128,0), interpolation="bilinear")
    axes[1,col].axis("off"); axes[1,col].set_title(f"p={prob:.2f}", fontsize=8)
axes[0,0].set_ylabel("원본", fontsize=10)
plt.suptitle("Grad-CAM — 위:원본 / 아래:관심영역 (모델이 보는 곳)", fontsize=13)
plt.tight_layout(); plt.savefig(FIG_DIR/"gradcam_samples.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- 단일 웨이퍼 추론 데모 함수 ---
@torch.no_grad()
def predict(wafer_u8, topk=3):
    x = to_tensor(wafer_u8).to(DEVICE)
    prob = net(x).softmax(1)[0].cpu().numpy()
    order = prob.argsort()[::-1][:topk]
    return [(CLASSES[i], float(prob[i])) for i in order]

# 임의 샘플로 데모
i = int(np.random.default_rng(1).choice(np.where(yte!=8)[0]))
print("실제 라벨:", CLASSES[yte[i]])
for name, p in predict(Xte[i]):
    print(f"  {name:10s} {p*100:5.1f}%")

plt.figure(figsize=(3,3)); plt.imshow(Xte[i], cmap="viridis", vmin=0, vmax=2)
plt.title(f"실제: {CLASSES[yte[i]]}"); plt.axis("off"); plt.show()